# Cell list

The sampling keeps its computational complexity managable by dividing the box up into cells that are just a bit larger than one particle diameter, which one could achieve, for example, by setting the number of divisions to $\left\lfloor\frac{L}{\sigma}\right\rfloor$ where $L$ is simulation box size and $\sigma$ particle diameter. The reason for this is that particle pairs that are not in the same cell or neighbouring cells are definitely out of their interaction range $\sigma$, making it redundant to compute their potential energy.

For this reason we can, in fact, also safely update cells that are two apart at the same time. Cells will be two apart if we give any given cell of an update group an immediate neighbourhood of cells (as skin, if you will) of thickness one, which will be $3^d$ in count; every part of this representative cell neighbourhood can be thought of as standing for the entire parallel-updating group. The number of divisions will have to divide $3$ for this to fit cleanly, so the number of divisions we choose is $$3\left\lfloor\frac{1}{3}\frac{L}{\sigma}\right\rfloor$$

# Limits to move magnitude

The limit to the size of moves is one particle diameter, since otherwise two particles from the same cell group could potentially interact


### Illustration
<div style="text-align: center;"><img src="cell_list_illustration.png" alt="Illustration of why half a particle diameter is the worst-case upper bound for move size" width="50%"></div>

In [10]:
import numpy as np
import numba as nb

In [11]:
from math import lgamma

@nb.njit
def B(d):
    '''-
    Returns the hyper-volume of the unit ball
    in d dimensions

    https://en.wikipedia.org/wiki/N-sphere#Volume_and_area
    '''
    gamma_of_1_plus_d_over_2 = np.exp(lgamma(1 + d/2))
    return np.pi**(d/2) / gamma_of_1_plus_d_over_2

In [12]:
@nb.njit
def u(r, E, sigma):
	'''
	pair potential for penetrable spheres or disks given a radius
	'''
	return E if r < sigma else (E/2 if r == sigma else 0)

In [ ]:
from random import shuffle



@nb.njit
def __inflate_flattened_index(flattened_index, spatial_dimension_count, indices_per_dimension):
    '''
    Computes the multiindex (i_0, i_1, ..., i_{d-1}) from the
    flattened cell index i_0 + i_1 * M + ... + i_{d-1} * M^(d-1)
    
    Args:
        flattened_index:
            the flattened index to inflate
        spatial_dimension_count:
            d from the above description
        indicies_per_dimension:
            M from the above description
    
    Returns:
        (i_0, i_1, ..., i_{d-1}) as an np.int64 array with d entries
    '''
    flattened_index_copy = int(flattened_index) # making a copy
    inflated_multiindex = np.zeros(spatial_dimension_count, dtype=np.int64) # output for later 
    for k in range(spatial_dimension_count):
        # let M = indices_per_dimension for brevity:
        # flattened_index = i_0 + i_1 * M + i_2 * M^2 + ... + i_{d-1} * M^(d-1), hence ...
        # ... flattened_index % M = i_0, making flattened_index - i_0 = i_1 * M + ..., such ...
        # ... that way may divide out M and repeat the process
        i_k = flattened_index_copy % indices_per_dimension
        inflated_multiindex[k] = i_k
        flattened_index_copy -= i_k
        # TODO dividing is nono. Can you avoid it?
        flattened_index_copy = int(flattened_index_copy / indices_per_dimension)
    return inflated_multiindex



@nb.njit
def __flatten_multiindex(multiindex, spatial_dimension_count, indices_per_dimension):
    '''
    Takes in a multiindex (i_0, i_1, ..., i_{d-1}) and flattens it into
    a cell index i_0 + i_1 * M + ... + i_{d-1} * M^(d-1). If any of the
    i_k exceeds M or falls below 0, i_k % M will be used in its stead.
    
    Args:
        multiindex:
            the (i_0, i_1, ..., i_{d-1}) to be flattened
        spatial_dimension_count:
            the number of spatial directions d implied
        indices_per_dimension:
            starting from 0, the maximum value that the i_k reach
    '''
    flattened_index = 0
    for l in range(spatial_dimension_count):
        flattened_index += int(
            # sparing user from having to watch out for in-range indices and ...
            # ... hence enable them to increment up and down freely
            (multiindex[l] % indices_per_dimension)
            * indices_per_dimension**l
        )
    return flattened_index



@nb.njit
def __repopulate_cell_list(
    x, cell_list, cell_list_size,
    divisions, simulation_box_size):
    '''
    takes in the array of positions and fills them into the cell list
    
    Args:
        bin_indices:
            list of bin indices of size spatial_dimension_count
        cell_list:
            the cell_list
    '''
    # retrieving particle number from x
    particle_number, spatial_dimension_count = x.shape
    # clearing all lists entries
    for i in range(cell_list_size):
        cell_list[i].clear()
    # bin indices are given by flooring
    bin_indices = np.floor(divisions * x / simulation_box_size).astype(np.int64) % divisions
    for k in range(particle_number):
        b_k = bin_indices[k]
        cell_list[__flatten_multiindex(b_k, spatial_dimension_count, divisions)].append(k)


def __neighbours_of():
    '''
    TODO IMPLEMENT
    '''
    pass



@nb.njit(parallel=True)
def run(
    overlap_energy, kB_times_temperature,
    packing_density, particle_number, spatial_dimension_count=3,
    simulation_box_size=1, move_count=1000):
    '''
    Executing the metropolis hastings sampling of
    the penetrable sphere system
    '''
    # computing the particle diameter corresponding to the ...
    # ... sought packing density
    volume = simulation_box_size**spatial_dimension_count
    number_density = particle_number / volume
    sigma = 2 * (
        packing_density / (
            B(spatial_dimension_count) * number_density)
            )**(1/spatial_dimension_count)
    # randomly placed particles
    x = np.random.rand(particle_number, spatial_dimension_count)
    # cell list for energy computation, where the number of divisions is ...
    # ... chosen to be multiples of three, in order to update cells later in ...
    # ... 3^d groups (hence the factors of 3 and 1/3 sandwiching the floor-operation)
    third_of_divisions = int(np.floor((1/3)*simulation_box_size / sigma)) # NOTE: used a lot later to inflate flattened indices
    divisions = int(3 * third_of_divisions)
    cell_list_size = divisions**spatial_dimension_count
    update_group_count = 3**spatial_dimension_count
    cells_per_update_group = int(cell_list_size / update_group_count)
    
    # functions need a fixed return type, so we cannot recursively construct ...
    # ... the cell list without some overloading magic. Hence I will just use ...
    # ... a flattened index
    cell_list = nb.typed.List([
        # the number of divisions per dimension
        nb.typed.List.empty_list(nb.types.int64)
        for _ in range(cell_list_size)
    ])
    for _ in range(move_count):
        # executing the sampling moves where we update ...
        # ... non-interacting cells in parallel, i.e. one of ...
        # ... 3^d groups of cells which are all out of ...
        # ... interaction range of each other; cells, separated ...
        # ... by three in any direction instead of one, are ...
        # ... updated, i.e. i_k = 3*n_k + g_k, where g=(g_0,...g_{d-1}) is the ...
        # ... index of the group, as represented by a virtual 3x3x...x3 ...
        # ... configuration of cells and n_k is the cell index
        for flattened_group_index in range(update_group_count):
            # inflating the update group index to (g_0,...,g_{d-1})
            g = __inflate_flattened_index(
                flattened_group_index,
                spatial_dimension_count,
                # will always be three along any given dimension: ...
				# ... the cell itself and its left and right neighbours
                3 
            )
            # we want the cell list to reflect previosly done moves before ...
            # ... updating the next group
            __repopulate_cell_list(
                x, cell_list, cell_list_size,
                divisions, simulation_box_size
            )
            # we now update all cells of group g in parallel
            for intra_group_index in nb.prange(cells_per_update_group):
                # figuring out the cell index
                cell_index = 0
                n = __inflate_flattened_index(
                    intra_group_index,
                    spatial_dimension_count,
                    # since update group g has only 1/3^d the number of cells in it, so a third ...
                    # ... of the divisions per dimension
                    third_of_divisions
                )
                # intra_group_index = n_0 + n_1 * M + n_2 * M**2 + ... + n_{d-1} * M**(d-1), hence ...
                # ... intra_group_index % M = n_0, making intra_group_index - n_0 = n_1 * M + ..., such ...
                # ... that way may divide out M and repeat the process. M here is the numer of cells ...
                # ... per update group
                for k in range(spatial_dimension_count):
                    cell_index += int((3 * n[k]  + g[k]) * divisions**k)
                # for all particles in the cell in question, we will now
                for i in cell_list[cell_index]:
                    x_i = x[i]
                    # grabbing the id of all neighbouring cells
                    u(0, overlap_energy, sigma) # TODO PER RELEVANT PAIR

        # shuffling all particles to not accidentally introduce ...
        # ... effects only due to the fixed update order
        shuffle(x)
        break # TODO REMOVE AGAIN


In [17]:
# test,
# TODO SET UP PYTEST
assert (__inflate_flattened_index(
    __flatten_multiindex(
		np.array([51,23,10],dtype=np.int64),
		3,
		100
	),
    3,
    100
) == np.array([51,23,10],dtype=np.int64)).all()

In [18]:
run(
    1, # overlap energy
    1, # kBT
    0.3, # packing density
    200, # 1000 # particle number
    move_count=100 # 1000
)

0,0,0
1,0,0
2,0,0
0,1,0
1,1,0
2,1,0
0,2,0
1,2,0
2,2,0
0,0,1
1,0,1
2,0,1
0,1,1
1,1,1
2,1,1
0,2,1
1,2,1
2,2,1
0,0,2
1,0,2
2,0,2
0,1,2
1,1,2
2,1,2
0,2,2
1,2,2
2,2,2


#### Non-Parallelized

compile time 1: `22.4s`

time 1: `11.1s`

#### Parallelized

compile time 2: `25.8`

time 2: `15.3s`